In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType


## acheck adls folders

In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/")

In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/")


### estrai tabelle da adls

In [0]:
# 1. Get the list of all files/folders in the path
raw_files = dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/")

# 2. Extract only the names
# We use .rstrip('/') because folder names in ADLS often end with a slash
tabelle = [file.name.rstrip('/') for file in raw_files]

# 3. Filter the list (if you only want specific tables)
# For example, if you want to skip the metadata table or hidden files
tabelle_clean = [t for t in tabelle if "weather" in t and not t.startswith("_")]

print(f"Tables found: {tabelle_clean}")

autoloader

In [0]:

source_base_path     = "abfss://source@storageaccountmeteo.dfs.core.windows.net/tables"
checkpoint_base_path = "abfss://source@storageaccountmeteo.dfs.core.windows.net/checkpoint"

queries = []

for t in tabelle_clean:
    print(f"Avvio ingestion per: {t}")

    s_path      = f"{source_base_path}/{t}/"
    schema_path = f"{checkpoint_base_path}/schema/{t}/"
    c_path      = f"{checkpoint_base_path}/checkpoint/{t}/"

    q = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(s_path)
        .writeStream
        .option("checkpointLocation", c_path)
        .trigger(availableNow=True)
        .toTable(f"catalogmeteo.bronze.{t}")
    )
    queries.append(q)

for q in queries:
    q.awaitTermination()

check format

In [0]:
spark.sql("DESCRIBE DETAIL catalogmeteo.bronze.weather_milan_2024").select("format").show()